In [3]:
import os
import pandas as pd
from pathlib import Path

RAW_DIR = os.path.join("..", "data", "raw", "1km")
OUT_DIR = os.path.join("..", "data", "processed")
os.makedirs(OUT_DIR, exist_ok=True)

S5P_PATH   = os.path.join(RAW_DIR, "jakarta_s5p_1km_2025.csv")
VIIRS_PATH = os.path.join(RAW_DIR, "jakarta_viirs_1km_2025.csv")
OUT_PATH = os.path.join(OUT_DIR, "jakarta_tanpa_Modis_1km_2025.csv")


<h4>Note : Ubah nama file datanya saja jika ingin ganti file input tahun 2023 & 2024 nya</h4>

In [7]:
def load_and_trim(path: str, feature_cols: list[str]) -> pd.DataFrame:
    df = pd.read_csv(path)

    required = {"lat", "lon"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"File {os.path.basename(path)} missing columns: {missing}")

    #hanya ambil lat, lon, dan feature cols 
    keep = ["lat", "lon"] + [c for c in feature_cols if c in df.columns]
    df = df[keep].copy()

    df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
    df["lon"] = pd.to_numeric(df["lon"], errors="coerce")
    for c in keep:
        if c not in ("lat", "lon"):
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # Drop baris yang lat/lon nya null
    df = df.dropna(subset=["lat", "lon"])
    
    return df

s5p = load_and_trim(S5P_PATH,   ["s5p_co", "s5p_no2", "s5p_o3", "s5p_so2"])
viirs = load_and_trim(VIIRS_PATH, ["viirs_ntl"])

print("S5P  :", s5p.shape)
print("VIIRS:", viirs.shape)


S5P  : (658, 6)
VIIRS: (658, 3)


In [5]:
def check_duplicates(df: pd.DataFrame, name: str):
    dup = df.duplicated(subset=["lat", "lon"]).sum()
    print(f"{name}: duplicates by (lat,lon) = {dup}")

check_duplicates(s5p, "S5P")
check_duplicates(viirs, "VIIRS")


S5P: duplicates by (lat,lon) = 0
VIIRS: duplicates by (lat,lon) = 0


In [6]:
merged = (
    s5p
    #.merge(modis, on=["lat", "lon"], how="inner")
    .merge(viirs, on=["lat", "lon"], how="inner")
)

ordered_cols = ["lat", "lon", "s5p_co", "s5p_no2", "s5p_o3", "s5p_so2", "viirs_ntl"]
ordered_cols = [c for c in ordered_cols if c in merged.columns]
merged = merged[ordered_cols].copy()

print("MERGED:", merged.shape)
merged.head()


MERGED: (658, 7)


,lat,lon,s5p_co,s5p_no2,s5p_o3,s5p_so2,viirs_ntl
0,-6.364564,106.886044,0.032389,0.000071,0.116598,0.000089,30.452932
1,-6.364564,106.895027,0.032423,0.000069,0.116594,0.000032,21.725202
2,-6.364564,106.912994,0.032622,0.000069,0.116500,0.000064,25.892045
3,-6.355581,106.796213,0.032594,0.000072,0.116421,0.000033,31.800545
4,-6.355581,106.805196,0.032570,0.000079,0.116511,0.000046,34.711155


In [6]:
merged.to_csv(OUT_PATH, index=False)
print("Saved to:", OUT_PATH)


Saved to: ..\data\processed\jakarta_tanpa_Modis_1km_2025.csv
